In [ ]:
encoder_path = "./model/encoder/model.onnx"
decoder_path = "./model/decoder/model.onnx"

ExecutionProvider="QNNExecutionProvider"

In [ ]:
# reference: https://learn.microsoft.com/en-us/windows/ai/new-windows-ml/tutorial?tabs=python#acquiring-the-model-and-preprocessing
import subprocess
import json
import sys
import os
import onnxruntime as ort

def register_execution_providers():
    worker_script = os.path.abspath('winml.py')
    print(worker_script)
    result = subprocess.check_output([sys.executable, worker_script], text=True)
    paths = json.loads(result)
    for item in paths.items():
        try:
            ort.register_execution_provider_library(item[0], item[1])
        except Exception as e:
            print(f"Failed to register execution provider {item[0]}: {e}")

register_execution_providers()

In [ ]:
from urllib import request

test_image_url = "https://github.com/facebookresearch/segment-anything/blob/main/notebooks/images/truck.jpg?raw=true"
test_image_name = "truck.jpg"

request.urlretrieve(test_image_url, test_image_name)

from IPython.display import Image, display

display(Image(filename=test_image_name))

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
from PIL import Image
from sam2_mask_generator import get_mask_ort

def add_ep_for_device(session_options, ep_name, device_type, ep_options=None):
    ep_devices = ort.get_ep_devices()
    for ep_device in ep_devices:
        if ep_device.ep_name == ep_name and ep_device.device.type == device_type:
            print(f"Adding {ep_name} for {device_type}")
            session_options.add_provider_for_devices([ep_device], {} if ep_options is None else ep_options)
            break


sess_options = ort.SessionOptions()

add_ep_for_device(sess_options, ExecutionProvider, ort.OrtHardwareDeviceType.CPU)

# Load image
raw_image = Image.open(test_image_name).convert("RGB")
input_box = [[[100, 300], [1750, 900]]]

# Load models
sess_ve = ort.InferenceSession(encoder_path, sess_options=sess_options)
sess_md = ort.InferenceSession(decoder_path, sess_options=sess_options)

sess_ve_inputs = sess_ve.get_inputs()
sess_md_inputs = sess_md.get_inputs()

ve_dtype = np.float32 if sess_ve_inputs[0].type == 'tensor(float)' else np.float16
md_dtype = np.float32 if sess_md_inputs[0].type == 'tensor(float)' else np.float16

# Get mask
mask = get_mask_ort(sess_ve, sess_md, raw_image, input_box, ve_dtype, md_dtype, sess_ve_inputs, sess_md_inputs)

# Save mask using PIL
mask_img = Image.fromarray(mask * 255)  # Convert binary mask to 0-255

from IPython.display import display
display(mask_img)